# Information Theory

信息论回答"不确定性有多少、如何度量"。熵、交叉熵、KL 散度是深度学习的日常工具——交叉熵损失就是名字里带"熵"的那个东西。


## 0. 环境配置与导入


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import torch
import matplotlib

matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB", "Arial Unicode MS", "Microsoft YaHei", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)


## 1. 自信息：事件的信息量


**自信息** $I(x) = -\log_2 p(x)$（单位 bit）：

- 概率越小的事件，发生时携带的信息越大（"太阳从西边出来" vs "今天下雨"）
- 取对数是强制的：两个独立事件的信息要**可加**，$I(x,y) = I(x) + I(y) \Leftrightarrow p(x,y) = p(x)p(y)$


## 2. 熵：平均信息量


$$H(p) = \mathbb{E}_{x\sim p}\big[-\log p(x)\big] = -\sum_x p(x)\log p(x)$$

熵 = 分布的**平均不确定性**。性质：

- $H \ge 0$；确定性分布 $H=0$
- 给定取值数 $K$，**均匀分布熵最大**（$\log K$）——最不确定


In [ ]:
def binary_entropy(p):
    p = np.clip(p, 1e-12, 1-1e-12)
    return -(p*np.log2(p) + (1-p)*np.log2(1-p))

ps = np.linspace(0, 1, 200)
plt.figure(figsize=(7, 4))
plt.plot(ps, [binary_entropy(p) for p in ps], 'k-')
plt.axvline(0.5, color='gray', ls='--', lw=0.8)
plt.text(0.52, 0.4, 'H 最大 = 1 bit @ p=0.5')
plt.xlabel('p（成功概率）'); plt.ylabel('H(p) (bit)')
plt.title('二值熵：p=0.5 时不确定性最大')
plt.grid(alpha=0.3)
print(f"H(0.5) = {binary_entropy(0.5):.4f} bit, H(0.1) = {binary_entropy(0.1):.4f} bit")


## 3. 交叉熵：用错误分布编码的代价


真实分布 $p$，但我们用 $q$ 去描述它（编码），平均需要的比特数：

$$H(p, q) = -\sum_x p(x)\log q(x)$$

**吉布斯不等式**：$H(p, q) \ge H(p)$，等号当且仅当 $q = p$。"用错误的分布编码，代价只会更高"。


In [ ]:
# 验证 H(p,q) ≥ H(p)，且 q 越接近 p 代价越小
p = np.array([0.5, 0.25, 0.25])
for q in [p, np.array([0.6, 0.2, 0.2]), np.array([0.9, 0.05, 0.05]), np.array([1/3, 1/3, 1/3])]:
    ce = -(p*np.log2(q)).sum()
    print(f"q={q}  H(p,q)={ce:.4f}  (H(p)={-(p*np.log2(p)).sum():.4f})")


## 4. KL 散度：两个分布的距离


$$D_{KL}(p \| q) = H(p, q) - H(p) = \sum_x p(x)\log\frac{p(x)}{q(x)}$$

KL 度量"用 q 近似 p 的额外代价"。性质：

- $D_{KL} \ge 0$（吉布斯不等式的直接推论），等号当且仅当 $p=q$
- **不对称**：$D_{KL}(p\|q) \ne D_{KL}(q\|p)$——所以不是严格的距离


In [ ]:
def kl(p, q):
    return np.sum(p*np.log(p/q))

p = np.array([0.6, 0.3, 0.1]); q = np.array([0.5, 0.35, 0.15])
print(f"KL(p‖q) = {kl(p, q):.5f}  ≥ 0")
print(f"KL(q‖p) = {kl(q, p):.5f}  ≥ 0（但数值不同 → 不对称）")
print(f"KL(p‖p) = {kl(p, p):.8f}  = 0")


## 5. 深度学习中的应用


- **交叉熵损失**：多分类 $\ell = -\log p_{y}$ 就是 $H(\text{one-hot 标签}, \text{模型分布})$。最小化 CE = 最小化 KL（标签分布固定时）
- **softmax + 交叉熵**：输出层构造概率分布 $q$，与真实标签 $p$（one-hot）做 KL
- **正则化视角**：Dropout/权重衰减可以理解为给模型分布加"熵惩罚"或"先验"（MAP）
- **生成模型**：VAE 的损失 = 重构项 + KL 项（让隐变量分布贴近先验，08 课）


In [ ]:
# 训练中的交叉熵：模型越准，CE 越接近 H(标签)
import torch.nn.functional as F
logits = torch.tensor([[2.5, 0.3, -1.2]], dtype=torch.float64)
target = torch.tensor([0])
print("CE 损失:", F.cross_entropy(logits, target).item())
print("模型分布:", torch.softmax(logits, dim=1).numpy().round(3))
# 当模型分布 = 标签分布（one-hot）时 CE = 0
print("完美预测 CE:", F.cross_entropy(torch.tensor([[100., 0., 0.]]), target).item())


## 课后练习


1. **证明吉布斯不等式**：用 $\log x \le x-1$ 证明 $D_{KL} \ge 0$。
2. **熵的范围**：$K=3$ 类分布，均匀熵最大为 $\log_2 3$，数值验证。
3. **交叉熵 vs KL**：证明 $H(p,q) = H(p) + D_{KL}(p\|q)$。
4. **softmax 温度**：画出不同温度下 softmax 的熵（衔接 08 课）。
5. **思考**：为什么训练用 CE 而不是直接最小化 0-1 错误率？（提示：可微性、梯度）
